In [3]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")

db = client["kopi_nusantara"]

koleksi_menu = db["menu"]

print("Koneksi MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'local', 'praktikum_bigdata']


In [4]:
# A. Membangun Katalog (Create)
# Menghapus data lama agar hasil konsisten
koleksi_menu.delete_many({})

# Memasukkan 12 dokumen menu
koleksi_menu.insert_many([
    {"nama_menu": "Espresso", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 30},
    {"nama_menu": "Americano", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 25},
    {"nama_menu": "Cappuccino", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 20},
    {"nama_menu": "Cafe Latte", "harga": 26000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 22},
    {"nama_menu": "Kopi Tubruk", "harga": 15000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 18},
    {"nama_menu": "Mocha latte", "harga": 30000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 15},
    {"nama_menu": "Nasi Goreng", "harga": 28000, "kategori": "Makanan", "pedas_level": 2, "stok": 15},
    {"nama_menu": "Mie Goreng", "harga": 24000, "kategori": "Makanan", "pedas_level": 3, "stok": 12},
    {"nama_menu": "Roti Bakar Cokelat", "harga": 18000, "kategori": "Makanan", "pedas_level": 0, "stok": 20},
    {"nama_menu": "Pisang Goreng", "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "stok": 25},
    {"nama_menu": "Kentang Goreng", "harga": 15000, "kategori": "Makanan", "pedas_level": 1, "stok": 17},
    {"nama_menu": "Mendoan", "harga": 10000, "kategori": "Makanan", "pedas_level": 0, "stok": 25}
])

print("Jumlah dokumen:", koleksi_menu.count_documents({}))

Jumlah dokumen: 12


In [10]:
# B. Kueri Bertingkat (Read) 
# 1. Menu Kopi dengan tingkat_kepahitan di atas 3
print("1. Kopi dengan tingkat kepahitan diatas 3:")
for p in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"  - {p['nama_menu']} (kepahitan {p['tingkat_kepahitan']})")

1. Kopi dengan tingkat kepahitan diatas 3:
  - Espresso (kepahitan 5)
  - Americano (kepahitan 4)
  - Kopi Tubruk (kepahitan 5)


In [11]:
# B. Kueri Bertingkat (Read) 
# 2. Menu dengan harga antara 15000 dan 30000
print("2. Menu dengan harga 15000 - 30000:")
for p in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"  - {p['nama_menu']} (Rp{p['harga']:,})")

2. Menu dengan harga 15000 - 30000:
  - Espresso (Rp18,000)
  - Americano (Rp20,000)
  - Cappuccino (Rp25,000)
  - Cafe Latte (Rp26,000)
  - Kopi Tubruk (Rp15,000)
  - Mocha latte (Rp30,000)
  - Nasi Goreng (Rp28,000)
  - Mie Goreng (Rp24,000)
  - Roti Bakar Cokelat (Rp18,000)
  - Pisang Goreng (Rp15,000)
  - Kentang Goreng (Rp15,000)


In [12]:
# B. Kueri Bertingkat (Read) 
# 3. Menu yang tidak punya field pedas_level
print("3. Menu tanpa field pedas_level:")
for p in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"  - {p['nama_menu']} ({p['kategori']})")

3. Menu tanpa field pedas_level:
  - Espresso (Kopi)
  - Americano (Kopi)
  - Cappuccino (Kopi)
  - Cafe Latte (Kopi)
  - Kopi Tubruk (Kopi)
  - Mocha latte (Kopi)


In [13]:
# C. Pembaruan Data (Update) 
# Mengubah status seluruh menu menjadi Tersedia
hasil_update = koleksi_menu.update_many(
    {},
    {"$set": {"status": "Tersedia"}}
)

# Mengubah status Espresso menjadi Habis
hasil_espresso = koleksi_menu.update_one(
    {"nama_menu": "Espresso"},
    {"$set": {"status": "Habis"}}
)

# Mengubah status Kentang Goreng menjadi Habis
hasil_nasi = koleksi_menu.update_one(
    {"nama_menu": "Kentang Goreng"},
    {"$set": {"status": "Habis"}}
)

print("Daftar menu setelah pembaruan status:")
for p in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"- {p['nama_menu']} | Status: {p['status']}")

Daftar menu setelah pembaruan status:
- Espresso | Status: Habis
- Americano | Status: Tersedia
- Cappuccino | Status: Tersedia
- Cafe Latte | Status: Tersedia
- Kopi Tubruk | Status: Tersedia
- Mocha latte | Status: Tersedia
- Nasi Goreng | Status: Tersedia
- Mie Goreng | Status: Tersedia
- Roti Bakar Cokelat | Status: Tersedia
- Pisang Goreng | Status: Tersedia
- Kentang Goreng | Status: Habis
- Mendoan | Status: Tersedia


In [14]:
# D. Integrasi ke Spark 
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count, col
import pandas as pd

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Ambil data dari MongoDB -> pandas
df_pandas = pd.DataFrame(list(koleksi_menu.find({})))
df_pandas["_id"] = df_pandas["_id"].astype(str)

# Field yang tidak dimiliki sebuah dokumen muncul sebagai NaN di pandas,
# diubah menjadi None agar terbaca sebagai null di Spark
df_pandas = df_pandas.astype(object).where(df_pandas.notna(), None)

# Konversi ke Spark DataFrame
df_spark = spark.createDataFrame(df_pandas)

# Samakan tipe kolom numerik
df_spark = df_spark \
    .withColumn("harga", col("harga").cast("int")) \
    .withColumn("stok", col("stok").cast("int")) \
    .withColumn("tingkat_kepahitan", col("tingkat_kepahitan").cast("int")) \
    .withColumn("pedas_level", col("pedas_level").cast("int"))

print("1. printSchema():")
df_spark.printSchema()
df_spark.show(truncate=False)

print("2. Rata-rata harga per kategori:")
df_spark.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_menu")
).show()

spark.stop()
print("SparkSession ditutup.")

26/11/01 11:03:55 WARN Utils: Your hostname, HP resolves to a loopback address: 127.0.1.1; using 192.168.0.101 instead (on interface wlo1)
26/11/01 11:03:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/01 11:03:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/01 11:03:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


1. printSchema():
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: integer (nullable = true)
 |-- stok: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: integer (nullable = true)



+------------------------+------------------+-----+--------+-----------------+----+--------+-----------+
|_id                     |nama_menu         |harga|kategori|tingkat_kepahitan|stok|status  |pedas_level|
+------------------------+------------------+-----+--------+-----------------+----+--------+-----------+
|6ae6b2d77161cbc9c44019ab|Espresso          |18000|Kopi    |5                |30  |Habis   |NULL       |
|6ae6b2d77161cbc9c44019ac|Americano         |20000|Kopi    |4                |25  |Tersedia|NULL       |
|6ae6b2d77161cbc9c44019ad|Cappuccino        |25000|Kopi    |3                |20  |Tersedia|NULL       |
|6ae6b2d77161cbc9c44019ae|Cafe Latte        |26000|Kopi    |2                |22  |Tersedia|NULL       |
|6ae6b2d77161cbc9c44019af|Kopi Tubruk       |15000|Kopi    |5                |18  |Tersedia|NULL       |
|6ae6b2d77161cbc9c44019b0|Mocha latte       |30000|Kopi    |3                |15  |Tersedia|NULL       |
|6ae6b2d77161cbc9c44019b1|Nasi Goreng       |28000|Maka

E. Kesimpulan
apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?
jawab:
## E. Kesimpulan

Tantangan utama saat mengonversi data MongoDB ke Spark DataFrame adalah ketidakseragaman field antar dokumen. Menu Kopi punya tingkat_kepahitan tetapi tidak punya pedas_level, sedangkan menu Makanan sebaliknya. Spark membutuhkan skema tabel yang seragam, jadi semua field harus dijadikan kolom, dan dokumen yang tidak memiliki field tertentu harus diisi nilai kosong. Masalah lain muncul saat data dibaca lewat pandas: field yang hilang menjadi NaN, dan kolom bilangan bulat berubah menjadi desimal. Selain itu, field id bertipe ObjectId tidak dikenali Spark. Cara mengatasinya: _id diubah menjadi string, NaN diubah menjadi None agar terbaca sebagai null di Spark, lalu tipe kolom numerik di-cast kembali ke integer. Dengan cara tersebut, data berhasil dikonversi menjadi Spark DataFrame dan dapat diolah untuk menampilkan struktur data serta menghitung rata-rata harga berdasarkan kategori.